# Workspace command validation
The workspace CLI orchestrates these functions: project creation, material registration, append-only revisions, one-passage replacement and the argument-graph page.

In [ ]:
import json, shutil, sys, tempfile
from pathlib import Path
root = Path.cwd()
while not (root / "knowledge_palace").is_dir():
    root = root.parent
sys.path.insert(0, str(root))
from knowledge_palace.tests.test_research import MINI
from knowledge_palace.workspace.material import new_material, register_material
from knowledge_palace.workspace.project import create_project, new_project
from knowledge_palace.workspace.research_helpers import project_context
from knowledge_palace.workspace.revision import latest, save_revision
from knowledge_palace.workspace.writing import replace_passage
temp = tempfile.TemporaryDirectory()
workspace = Path(temp.name) / "workspace"
project_dir = create_project(workspace, new_project("demo-paper", "paper", "hydrology researchers"))
sorted(path.name for path in project_dir.iterdir())

In [ ]:
(project_dir / "materials" / "draft.md").write_text("Draft introduction.", encoding="utf-8")
registry = register_material(project_dir, new_material("draft-1", "materials/draft.md", "draft"))
json.loads(registry.read_text(encoding="utf-8"))

In [ ]:
first = save_revision(project_dir, "introduction", "Echo cancellation is hard. It fails in rooms.\n")
revised = replace_passage(first.read_text(encoding="utf-8"), "It fails in rooms.", "It degrades under reverberation.")
second = save_revision(project_dir, "introduction", revised)
latest(project_dir, "introduction"), second.read_text(encoding="utf-8")

In [ ]:
context = project_context(workspace, "demo-paper", MINI)
assert context["materials"][0]["id"] == "material:draft-1"
context["project"], context["materials"]

In [ ]:
from knowledge_palace.tests.test_argument_graph import GRAPH
from knowledge_palace.workspace.argument_graph import changed_paragraphs, read_view, render_view
page = project_dir / "outline" / "graph-introduction.html"
page.write_text(render_view(GRAPH), encoding="utf-8")
graph, written = read_view(page.read_text(encoding="utf-8"))
graph["paragraphs"][1]["point"] = "a drift-aware canceller"
changed_paragraphs(graph, written)

In [ ]:
temp.cleanup()